In [1]:
import glob
import yaml
import spacy
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
import torch
from tqdm import tqdm
import weaviate
from collections import defaultdict
import json
import re
from bs4 import BeautifulSoup
from huggingface_hub import hf_hub_download, list_repo_files

/Applications/anaconda3/envs/fasthtml/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
model = SentenceTransformer('sentence-transformers/LaBSE')

/Applications/anaconda3/envs/fasthtml/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [11]:
repo_id = "placingholocaust/cleaned-html"
files = list_repo_files(repo_id, repo_type="dataset")[2:]

In [12]:
files[:2]

['RG-50.030.0001_trs_en_cleaned.html', 'RG-50.030.0002_trs_en_cleaned.html']

In [3]:
# for filename in files:
#     hf_hub_download(repo_id=repo_id, filename=filename, local_dir="../data/06_ner_cleaned", repo_type="dataset")

In [9]:
local_files = glob.glob("../data/07_final/*.html")
with open(local_files[0], "r", encoding="utf-8") as f:
    data = f.read()
len(local_files)

979

In [10]:
print(data[:5000])

---
layout: transcript
interviewee: alice none jakubovic
rg_number: rg-50.030.0469
pdf_url: https://collections.ushmm.org/oh_findingaids/rg-50.030.0469_trs_en.pdf
ushmm_url: https://collections.ushmm.org/search/catalog/irn511521
gender: f
birth_date: 1922-05-11
birth_year: 1922.0
place_of_birth: prešov
country: slovakia
experience_group: survivor
ghetto(s)_encyclopedia: none
ghetto: none
camp(s)_encyclopedia: auschwitz,ravensbruck
camp: none
non_ss_camp: none
region: none
needs_research: none
data_entry: cl
accession: 2002.237
revisit: none
tags: transcripts
---
---
layout: transcript
interviewee: alice none jakubovic
rg_number: rg-50.030.0469
pdf_url: https://collections.ushmm.org/oh_findingaids/rg-50.030.0469_trs_en.pdf
ushmm_url: https://collections.ushmm.org/search/catalog/irn511521
gender: f
birth_date: 1922-05-11
birth_year: 1922.0
place_of_birth: prešov
country: slovakia
experience_group: survivor
ghetto(s)_encyclopedia: none
ghetto: none
camp(s)_encyclopedia: auschwitz,ravensbr

In [16]:
def process_testimony(data):
    soup = BeautifulSoup(data, 'html.parser')
    metadata_include = ["rg_number", "interviewee", "gender", "birth_year", "experience_group", "birth_country"]

    def get_metadata(soup):
        # Extract the YAML front matter from the HTML
        front_matter_text = soup.text.split('---\n', 2)[1]
        metadata = yaml.safe_load('---\n' + front_matter_text)

        # Only keep metadata found in metadata_include
        filtered_metadata = {k: metadata[k] for k in metadata_include if k in metadata}
        filtered_metadata["birth_country"] = metadata.get("country", "")
        return filtered_metadata

    def extract_windows(dialogue_tag, metadata):
        p_tag = dialogue_tag.find('p')
        sentences = p_tag.find_all("sentence")
        p_text = " ".join([sent.text for sent in sentences])
        windows = []
        window_size = 3
        sentence_ids = [sent['id'] for sent in sentences]
        
        if len(sentences) < window_size:
            window_texts = " ".join([sent.text for sent in sentences])
            window = {
                'sentence_ids': sentence_ids,
                'text': window_texts,
                'labels': count_labels(sentences)
            }
            window["category"] = "question" if dialogue_tag.get('class') == ['Question'] else "answer"
            window.update(metadata)
            windows.append(window)
        else:
            for i in range(len(sentences) - window_size + 1):
                window_sentences = sentences[i:i+window_size]
                window_texts = " ".join([sent.text for sent in window_sentences])
                window = {
                    'sentence_ids': sentence_ids[i:i+window_size],
                    'text': window_texts,
                    'labels': count_labels(window_sentences)
                }
                window["category"] = "question" if dialogue_tag.get('class') == ['Question'] else "answer"
                window.update(metadata)
                windows.append(window)

        return windows

    def count_labels(sentences):
        label_counters = {
            'POPULATED_PLACE': 0, 'BUILDING': 0, 'COUNTRY': 0, 'SPATIAL_OBJECT': 0, 
            'DLF': 0, 'INTERIOR_SPACE': 0, 'ENV_FEATURES': 0, 'REGION': 0, 
            'NPIP': 0, "COUNTRY": 0,
        }
        for sentence in sentences:
            for label in label_counters:
                label_counters[label] += len(sentence.find_all("span", {"class": label}))
        
        return label_counters

    metadata = get_metadata(soup)
    all_windows = []
    for dialogue_tag in soup.find_all('dialogue'):
        windows = extract_windows(dialogue_tag, metadata)
        all_windows.extend(windows)

    sentence_embeddings = model.encode([text["text"] for text in all_windows])

    combined_data = []

    for i, window in enumerate(all_windows):
        combined_dict = {
            "sentence_ids": window['sentence_ids'],
            "text": window['text'],
            "embedding": sentence_embeddings[i],
            "category": window["category"]
        }
        combined_dict.update(window['labels'])
        
        label_map = {"rg_number": "rg", "interviewee": "full_name"}
        for label in metadata_include:
            new_label = label_map.get(label, label)
            combined_dict[new_label] = window[label]
        combined_data.append(combined_dict)
    return combined_data

In [22]:
import os
from tqdm import tqdm
import pandas as pd

for file in tqdm(local_files):
    output_file = file.split("/")[-1].replace("_cleaned.html", ".parquet")
    output_path = f"../data/09_parquet/{output_file}"
    

    # Check if the output file already exists
    if os.path.exists(output_path):
        continue  # Skip this file and move to the next one
    
    with open(file, "r", encoding="utf-8") as f:
        data = f.read()
        data = data.replace("|", "I")
        result = process_testimony(data)
        df = pd.DataFrame(result)
        df['birth_year'] = df['birth_year'].replace('none', pd.NA)
        df.to_parquet(output_path)

100%|██████████| 979/979 [42:22<00:00,  2.60s/it]
